# Cvičné úlohy — řešení

Ke každé úloze: seznam chyb, opravený kód, rozšíření a výsledky ladění.
**Dívej se sem až po vlastním pokusu** — jinak si vyrobíš pocit znalosti bez znalosti.

---

## Úloha 1 — Bankovní účet

*Archetyp: chybějící self, odsazení metod (katalog #1, #2)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | `def __init__(majitel, ...)` — **chybí `self`**. `majitel` je ve skutečnosti instance. |
| 2 | sémantická | `self.majitel = majitel` uvnitř → `NameError`, protože `self` neexistuje. |
| 3 | sémantická | `vloz` a `vyber` jsou **odsazené uvnitř `__init__`** — lokální funkce, ne metody. |
| 4 | sémantická | I ony nemají `self`. |
| 5 | návrhová | Žádná validace — jde vložit záporná částka i vybrat víc, než je na účtu. |

**Co kód udělá:** `Ucet("Anna", 1000)` spadne na `TypeError: takes from 1 to 2 positional
arguments but 3 were given`. Kdyby se předal jen jeden argument, spadlo by to na `NameError: self`.

### Opravené a rozšířené řešení

In [ ]:
class Ucet:
    """Bankovní účet s vkladem, výběrem a historií transakcí."""

    def __init__(self, majitel, zustatek=0):
        if not isinstance(majitel, str) or not majitel.strip():
            raise ValueError(f"majitel musí být neprázdný řetězec, dostal jsem {majitel!r}")
        if zustatek < 0:
            raise ValueError(f"počáteční zůstatek nesmí být záporný: {zustatek}")

        self.majitel = majitel.strip()
        self.zustatek = zustatek
        self.historie = []                    # atribut INSTANCE, ne třídy!

    def vloz(self, castka):
        """Vloží částku na účet. Vrací nový zůstatek."""
        if castka <= 0:
            raise ValueError(f"vklad musí být kladný, dostal jsem {castka}")
        self.zustatek += castka
        self.historie.append(("vklad", castka))
        return self.zustatek

    def vyber(self, castka):
        """Vybere částku. Vyhazuje ValueError, není-li dost prostředků."""
        if castka <= 0:
            raise ValueError(f"výběr musí být kladný, dostal jsem {castka}")
        if castka > self.zustatek:
            chybi = castka - self.zustatek
            raise ValueError(f"nedostatek prostředků: chybí {chybi} Kč")
        self.zustatek -= castka
        self.historie.append(("výběr", castka))
        return self.zustatek

    def vypis_historii(self):
        """Vrací formátovaný přehled transakcí."""
        if not self.historie:
            return "Žádné transakce."
        radky = [f"  {typ:6} {castka:>8} Kč" for typ, castka in self.historie]
        return "\n".join([f"Historie účtu {self.majitel}:"] + radky)

    def __str__(self):
        return f"{self.majitel}: {self.zustatek} Kč"

    def __repr__(self):
        return f"Ucet({self.majitel!r}, {self.zustatek})"

In [ ]:
u = Ucet("Anna", 1000)
u.vloz(500)
u.vyber(200)
print(u)                    # Anna: 1300 Kč
print(repr(u))              # Ucet('Anna', 1300)
print(u.vypis_historii())

print()
# každý účet má VLASTNÍ historii
b = Ucet("Bob")
print("Bobova historie:", b.historie)     # [] — neprosákla Annina

for popis, volani in [
    ("záporný vklad",  lambda: u.vloz(-100)),
    ("nulový výběr",   lambda: u.vyber(0)),
    ("výběr nad rámec", lambda: u.vyber(99999)),
    ("prázdný majitel", lambda: Ucet("  ")),
]:
    try:
        volani()
        print(f"{popis:18}: PROŠLO (nemělo!)")
    except ValueError as e:
        print(f"{popis:18}: {e}")

### Výsledky ladění

- **Funguje:** `Ucet("Anna", 1000)`, vklad 500, výběr 200 → zůstatek 1300. Ověřeno ručně.
- **Funguje:** `__str__` dá `Anna: 1300 Kč`, `__repr__` dá `Ucet('Anna', 1300)`.
- **Funguje:** historie zachytí obě transakce ve správném pořadí.
- **Klíčové:** `self.historie = []` je v `__init__`, ne v těle třídy — každý účet má vlastní.
  Ověřeno: Bobova historie je prázdná i po Anniných transakcích.
- **Hraniční:** záporný i nulový vklad/výběr → `ValueError`. Výběr nad rámec hlásí, **kolik chybí**.
- **Hraniční:** prázdný majitel i samé mezery → `ValueError`.
- **Omezení:** částky jsou `int`/`float` — pro reálné peníze by se použil `Decimal`
  kvůli zaokrouhlovacím chybám.
- **Omezení:** třída není vláknově bezpečná; při souběžném přístupu by zůstatek nesouhlasil.

### Na co se doptají

- **Jak poznáš chybějící `self`?** Podle hlášky `takes N positional arguments but N+1 were given`.
  Python předal instanci navíc.
- **Proč se metody nesmí odsadit dovnitř `__init__`?** Staly by se lokálními funkcemi,
  které se při každém vytvoření instance nadefinují a zahodí. Navenek by třída metodu neměla.
- **Proč `self.historie = []` v `__init__` a ne `historie = []` v těle třídy?** Atribut třídy
  by byl **sdílený všemi instancemi** — všechny účty by měly společnou historii.
- **Musí se `self` jmenovat `self`?** Ne, je to konvence. Python předává instanci podle **pozice**,
  ne podle jména. Ale pojmenovat ho jinak je matoucí — přesně to dělá chybný kód v zadání.

---

## Úloha 2 — Počítadlo návštěv

*Archetyp: atribut třídy vs. instance (katalog #13, #3)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`navstevy = []` je atribut třídy** — sdílený **všemi** instancemi. Návštěvy všech stránek padají do jednoho seznamu. |
| 2 | sémantická | `return len(navstevy)` — **holý název bez `self.`** → `NameError`. |

**Co kód udělá:** spadne na `NameError: name 'navstevy' is not defined`.
Kdybys opravil jen tohle, dostal bys `home: 3` a `about: 3` — **obě stránky sdílejí seznam**.

**Tohle je stejná past jako mutable default argument** z [okruhu 1](../../01-programovani-funkce-a-cykly/):
měnitelný objekt vytvořený jednou a sdílený všemi.

### Opravené a rozšířené řešení

In [ ]:
class Stranka:
    """Stránka webu s evidencí návštěv."""

    celkem_stranek = 0          # atribut TŘÍDY — tady je to SPRÁVNĚ, počítá instance

    def __init__(self, url):
        if not isinstance(url, str) or not url.startswith("/"):
            raise ValueError(f"URL musí být řetězec začínající '/', dostal jsem {url!r}")

        self.url = url
        self.navstevy = []                  # atribut INSTANCE — každá stránka vlastní
        Stranka.celkem_stranek += 1         # přes JMÉNO TŘÍDY, ne self!

    def navstiv(self, cas):
        self.navstevy.append(cas)

    def pocet(self):
        return len(self.navstevy)           # oprava: self.

    def __len__(self):
        return len(self.navstevy)

    def __contains__(self, cas):
        return cas in self.navstevy

    def __str__(self):
        return f"{self.url} ({len(self.navstevy)} návštěv)"

    def __repr__(self):
        return f"Stranka({self.url!r})"

In [ ]:
home = Stranka("/")
about = Stranka("/about")

home.navstiv("10:00")
home.navstiv("10:05")
about.navstiv("11:00")

print("home: ", home.pocet(), "| len():", len(home))     # 2 2
print("about:", about.pocet(), "| len():", len(about))   # 1 1
print("seznamy jsou nezávislé:", home.navstevy is not about.navstevy)

print()
print("'10:00' in home: ", "10:00" in home)     # True
print("'10:00' in about:", "10:00" in about)    # False
print("celkem stránek:", Stranka.celkem_stranek)  # 2
print(home)                                       # / (2 návštěv)

try:
    Stranka("bez-lomitka")
except ValueError as e:
    print("neplatné URL:", e)

### Výsledky ladění

- **Funguje:** `home` má 2 návštěvy, `about` 1 — každá stránka má **vlastní** seznam.
  Ověřeno explicitně přes `is not`. Původní kód by dal 3 a 3.
- **Funguje:** `len(stranka)` i `"10:00" in stranka` díky `__len__` a `__contains__`.
- **Funguje:** `celkem_stranek` je 2 — tohle je **legitimní** použití atributu třídy,
  protože počet instancí je opravdu společná vlastnost.
- **Klíčový rozdíl:** `self.navstevy` (instance, vlastní) vs. `Stranka.celkem_stranek`
  (třída, sdílený). **Ta samá syntaxe, opačný záměr** — proto se na to komise ptá.
- **Hraniční:** URL bez lomítka i nesprávný typ → `ValueError`. Stránka bez návštěv → `len` je 0.
- **Omezení:** `celkem_stranek` se zvyšuje přes `Stranka.celkem_stranek`, ne `self.` —
  `self.celkem_stranek += 1` by vytvořilo **nový atribut instance** a atribut třídy by zůstal na 0.
- **Omezení:** čas je řetězec, neověřuje se formát. Reálně by se použil `datetime`.

### Na co se doptají

- **Kdy je atribut třídy správně a kdy chyba?** Správně u **neměnných sdílených konstant**
  (`colors = [...]`, `MAX = 100`) a u počítadel instancí. Chyba u **měnitelných dat instance**.
- **Co se stane při `self.celkem_stranek += 1`?** Přečte se hodnota z třídy, přičte se 1
  a výsledek se uloží jako **nový atribut instance**. Atribut třídy zůstane nezměněný.
  Proto se musí psát `Stranka.celkem_stranek += 1`.
- **Proč `len(navstevy)` bez `self.` nefunguje?** Tělo třídy není jmenný prostor metod.
  Uvnitř metody vidíš jen lokální proměnné, globální a builtiny — atributy jen přes `self`.
- **Jak souvisí s mutable default argumentem?** Stejný princip: měnitelný objekt vytvořený
  **jednou** při definici a sdílený všemi následnými použitími.

---

## Úloha 3 — Zlomek

*Archetyp: __str__ vs. __repr__ (katalog #7, #18)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`__str__` nemá `return`.** Vytvoří f-string a zahodí ho → vrací `None`. |
| 2 | runtime | `print(z)` spadne: `TypeError: __str__ returned non-string (type NoneType)`. |
| 3 | návrhová | Chybí `__repr__` — výpis v seznamu dá `<__main__.Zlomek object at 0x...>`. |
| 4 | návrhová | Nulový jmenovatel projde a vytvoří nesmyslný objekt. |

**Co kód udělá:** spadne na `TypeError: __str__ returned non-string`.
Tahle hláška je dost specifická — když ji uvidíš, hledej chybějící `return` v `__str__`.

### Opravené a rozšířené řešení

In [ ]:
import math


class Zlomek:
    """Zlomek v základním tvaru se znaménkem v čitateli."""

    def __init__(self, citatel, jmenovatel):
        if not all(isinstance(x, int) and not isinstance(x, bool)
                   for x in (citatel, jmenovatel)):
            raise TypeError("čitatel i jmenovatel musí být celá čísla")
        if jmenovatel == 0:
            raise ValueError("jmenovatel nesmí být nula")

        # znaménko vždy do čitatele:  1/-2  ->  -1/2
        if jmenovatel < 0:
            citatel, jmenovatel = -citatel, -jmenovatel

        # zkrácení na základní tvar
        delitel = math.gcd(abs(citatel), jmenovatel) or 1
        self.citatel = citatel // delitel
        self.jmenovatel = jmenovatel // delitel

    def __str__(self):
        return f"{self.citatel}/{self.jmenovatel}"      # oprava: return!

    def __repr__(self):
        return f"Zlomek({self.citatel}, {self.jmenovatel})"

    @property
    def hodnota(self) -> float:
        """Desetinná hodnota zlomku."""
        return self.citatel / self.jmenovatel

    def __eq__(self, other):
        if not isinstance(other, Zlomek):
            return NotImplemented
        return (self.citatel, self.jmenovatel) == (other.citatel, other.jmenovatel)

    def __hash__(self):
        return hash((self.citatel, self.jmenovatel))

In [ ]:
z = Zlomek(3, 4)
print(z)                       # 3/4        — __str__
print([z, z])                  # [Zlomek(3, 4), Zlomek(3, 4)]   ← __repr__!
print(f"zlomek je {z}")        # zlomek je 3/4
print(f"{z.hodnota=}")         # 0.75       — property, bez závorek

print()
print("zkrácení:")
for c, j in [(6, 8), (10, 5), (1, -2), (-4, -8), (0, 5)]:
    print(f"  {c:>3}/{j:<3} -> {Zlomek(c, j)}")

print()
print("rovnost po zkrácení:", Zlomek(6, 8) == Zlomek(3, 4))   # True

try:
    Zlomek(1, 0)
except ValueError as e:
    print("nulový jmenovatel:", e)

### Výsledky ladění

- **Funguje:** `print(z)` dá `3/4`, `print([z, z])` dá `[Zlomek(3, 4), Zlomek(3, 4)]`.
  **Rozdíl je vidět:** v seznamu se volá `__repr__`, ne `__str__`.
- **Funguje:** zkrácení `6/8 → 3/4`, `10/5 → 2/1`, `-4/-8 → 1/2`, `0/5 → 0/1`.
- **Funguje:** znaménko jde do čitatele — `1/-2` se uloží jako `-1/2`.
- **Funguje:** `Zlomek(6,8) == Zlomek(3,4)` je `True`, protože oba se zkrátí na stejný tvar.
- **Funguje:** `hodnota` je property, čte se bez závorek → `0.75`.
- **Hraniční:** nulový jmenovatel → `ValueError`. `0/5` se zkrátí na `0/1` (`gcd(0,5)` je 5).
- **Omezení:** `math.gcd(0, 0)` je 0, což by dělilo nulou — ošetřeno přes `or 1`,
  ale ta větev je nedosažitelná, protože nulový jmenovatel se odmítne dřív.
- **Omezení:** zlomek se nedá sčítat ani násobit — chybí `__add__`, `__mul__`.
  Zadání to nežádalo, ale u obhajoby je to přirozená doptávka.

### Na co se doptají

- **Kdy se volá `__str__` a kdy `__repr__`?** `__str__` při `print()` a `str()`.
  `__repr__` v konzoli, při `repr()` a **při výpisu uvnitř kolekce**.
- **Co když definuji jen jedno?** Když jen `__repr__`, použije se i pro `str()` (fallback).
  Opačně to neplatí — proto piš `__repr__` vždycky.
- **Jak má `__repr__` vypadat?** Ideálně jako **kód, kterým objekt vytvoříš**: `Zlomek(3, 4)`.
  Pak jde `eval(repr(z))` a dostaneš rovnocenný objekt.
- **Proč zkracovat už v konstruktoru?** Aby byla rovnost jednoduchá —
  `6/8` a `3/4` mají po zkrácení stejné atributy, takže `__eq__` může porovnat přímo je.
- **Co dělá `math.gcd`?** Největší společný dělitel (Euklidův algoritmus).
  Teorie v [SZZTP okruh 11](../../../SZZTP/11-rekurence-asymptotika/).

---

## Úloha 4 — Barva v paletě

*Archetyp: __eq__ bez __hash__ (katalog #6, #15)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Definice `__eq__` vypnula `__hash__`.** Vložení do `set` → `TypeError: unhashable type: 'Barva'`. |
| 2 | sémantická | `__eq__` **nekontroluje typ** — `Barva(1,2,3) == "něco"` spadne na `AttributeError`. |
| 3 | návrhová | Žádná validace rozsahu — `Barva(999, -5, 0)` projde. |

**Co kód udělá:** porovnání projde (`True`), ale `{cervena, zelena}` spadne na
`TypeError: unhashable type: 'Barva'`.

**Proč Python `__hash__` vypíná:** platí pravidlo „co je si rovné, musí mít stejný hash".
Když předefinuješ rovnost, Python neví, jak z ní hash odvodit — tak ho radši zruší,
aby ses nedostal do stavu, kdy se ti objekt v množině „ztratí".

### Opravené a rozšířené řešení

In [ ]:
class Barva:
    """RGB barva se složkami 0-255."""

    def __init__(self, r, g, b):
        for nazev, hodnota in (("r", r), ("g", g), ("b", b)):
            if isinstance(hodnota, bool) or not isinstance(hodnota, int):
                raise TypeError(f"složka {nazev} musí být celé číslo, dostal jsem {hodnota!r}")
            if not 0 <= hodnota <= 255:
                raise ValueError(f"složka {nazev} je mimo rozsah 0-255: {hodnota}")
        self.r, self.g, self.b = r, g, b

    def __eq__(self, other):
        if not isinstance(other, Barva):
            return NotImplemented           # oprava 2: neháže AttributeError
        return (self.r, self.g, self.b) == (other.r, other.g, other.b)

    def __hash__(self):
        return hash((self.r, self.g, self.b))   # oprava 1: ze STEJNÝCH atributů jako __eq__

    def __repr__(self):
        return f"Barva({self.r}, {self.g}, {self.b})"

    def __str__(self):
        return self.hex

    @property
    def jas(self) -> float:
        """Vnímaný jas 0-1 podle luminance koeficientů."""
        return (0.299 * self.r + 0.587 * self.g + 0.114 * self.b) / 255

    @property
    def je_tmava(self) -> bool:
        return self.jas < 0.5

    @property
    def hex(self) -> str:
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"

In [ ]:
cervena = Barva(255, 0, 0)
zelena = Barva(0, 255, 0)
bila = Barva(255, 255, 255)
cerna = Barva(0, 0, 0)

print("rovnost:", cervena == Barva(255, 0, 0))     # True
print("s cizím typem:", cervena == "červená")      # False, ne výjimka

paleta = {cervena, zelena, Barva(255, 0, 0)}       # duplicita splyne
print("paleta:", paleta, "| velikost:", len(paleta))   # 2

# jako klíč slovníku
nazvy = {cervena: "červená", zelena: "zelená"}
print("klíč slovníku:", nazvy[Barva(255, 0, 0)])   # 'červená'

print()
for b in [cervena, zelena, bila, cerna]:
    print(f"  {b.hex}  jas={b.jas:.3f}  tmavá={b.je_tmava}")

print()
for popis, args in [("mimo rozsah", (999, 0, 0)), ("záporná", (-5, 0, 0))]:
    try:
        Barva(*args)
        print(f"{popis:14}: PROŠLO (nemělo!)")
    except (ValueError, TypeError) as e:
        print(f"{popis:14}: {type(e).__name__}: {e}")

### Výsledky ladění

- **Funguje:** `{cervena, zelena, Barva(255,0,0)}` má **2 prvky** — duplicita splynula
  díky `__hash__`. Původní kód tady padal na `TypeError: unhashable type`.
- **Funguje:** barva jde použít i jako **klíč slovníku** — `nazvy[Barva(255,0,0)]` vrátí `'červená'`,
  přestože je to jiný objekt než ten vložený.
- **Funguje:** `cervena == "červená"` vrací `False` místo `AttributeError` díky `NotImplemented`.
- **Funguje:** `hex` dává `#ff0000`, `#00ff00`, `#ffffff`, `#000000` — formát `:02x` doplní nulu.
- **Funguje:** jas červené je 0.299, zelené 0.587, bílé 1.0, černé 0.0. Zelená je **vnímaná
  jako světlejší než červená**, i když má stejnou číselnou hodnotu — proto ty koeficienty.
- **Hraniční:** složka 999, −5 i `"255"` → výjimka s uvedením, která složka je špatně.
  `True` je odmítnut zvlášť (dědí z `int`).
- **Omezení:** hash je počítaný ze stejné trojice jako rovnost — kdyby se složky daly měnit,
  objekt by se v množině „ztratil". Proto by správně měly být **read-only property**.

### Na co se doptají

- **Proč definice `__eq__` vypne `__hash__`?** Platí invariant „rovné objekty mají stejný hash".
  Python neumí z tvého `__eq__` hash odvodit, takže ho zruší, aby nevznikl nekonzistentní stav.
- **Z čeho počítat hash?** **Ze stejných atributů jako `__eq__`**, typicky `hash(n-tice)`.
- **Co se stane, když se atributy použité v hashi změní?** Objekt se v množině „ztratí" —
  hledá se na nové pozici, ale leží na staré. Proto mají být hashovatelné objekty **neměnné**.
- **K čemu je `NotImplemented`?** Signál „neumím se porovnat s tímhle typem".
  Python zkusí opačné porovnání a nakonec vrátí `False`. Lepší než výjimka.
- **Jaký je rozdíl mezi `NotImplemented` a `NotImplementedError`?** První je **hodnota**,
  kterou vracíš z porovnání. Druhá je **výjimka** pro nehotové metody. Snadno se pletou.

---

## Úloha 5 — Teploměr

*Archetyp: rekurze v property, chybějící setter (katalog #9, #8)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Getter vrací `self.celsia`** — což volá **sám sebe** → `RecursionError`. |
| 2 | sémantická | **Setter přiřazuje do `self.celsia`** — volá sám sebe → totéž. |

**Co kód udělá:** `Teplomer(20)` spadne na `RecursionError: maximum recursion depth exceeded`.

**Proč:** `self.celsia` **je** ta property. Když ji getter čte, spustí se getter znovu.
Řešení: property musí ukládat do **jinak pojmenovaného** atributu, konvenčně
s podtržítkem — `self._celsia`. To už není property, ale obyčejný atribut.

### Opravené a rozšířené řešení

In [ ]:
class Teplomer:
    """Teploměr s převody a validací proti absolutní nule."""

    ABSOLUTNI_NULA = -273.15

    def __init__(self, celsia):
        self.celsia = celsia            # projde SETTEREM, takže validace platí i tady

    @property
    def celsia(self):
        return self._celsia             # oprava: podtržítko, ne self.celsia!

    @celsia.setter
    def celsia(self, hodnota):
        if isinstance(hodnota, bool) or not isinstance(hodnota, (int, float)):
            raise TypeError(f"teplota musí být číslo, dostal jsem {type(hodnota).__name__}")
        if hodnota < self.ABSOLUTNI_NULA:
            raise ValueError(f"pod absolutní nulou: {hodnota} < {self.ABSOLUTNI_NULA}")
        self._celsia = hodnota          # oprava: ukládá do _celsia

    @property
    def fahrenheity(self):
        return self._celsia * 9 / 5 + 32

    @fahrenheity.setter
    def fahrenheity(self, hodnota):
        self.celsia = (hodnota - 32) * 5 / 9    # přepočte a projde validací

    @property
    def kelviny(self):
        """Jen ke čtení — nemá setter."""
        return self._celsia - self.ABSOLUTNI_NULA

    @property
    def stav(self):
        if self._celsia < 0:
            return "mráz"
        elif self._celsia < 15:
            return "chladno"
        elif self._celsia < 25:
            return "teplo"
        return "horko"

    def __str__(self):
        return f"{self._celsia:.1f} °C ({self.fahrenheity:.1f} °F, {self.stav})"

    def __repr__(self):
        return f"Teplomer({self._celsia})"

In [ ]:
t = Teplomer(20)
print(t)                          # 20.0 °C (68.0 °F, teplo)
print(f"{t.kelviny=:.2f}")        # 293.15

t.celsia = 30
print("po zápisu Celsia:  ", t)

t.fahrenheity = 32                # zápis ve F -> přepočte na 0 °C
print("po zápisu Fahrenheita:", t)

print()
for c in [-10, 5, 20, 35]:
    print(f"  {c:>4} °C -> {Teplomer(c).stav}")

print()
for popis, akce in [
    ("pod absolutní nulou", lambda: Teplomer(-300)),
    ("F pod absolutní nulou", lambda: setattr(t, "fahrenheity", -500)),
    ("není číslo",          lambda: Teplomer("teplo")),
    ("zápis do kelvinů",    lambda: setattr(t, "kelviny", 300)),
]:
    try:
        akce()
        print(f"{popis:22}: PROŠLO (nemělo!)")
    except (ValueError, TypeError, AttributeError) as e:
        print(f"{popis:22}: {type(e).__name__}: {e}")

### Výsledky ladění

- **Funguje:** `Teplomer(20)` → `20.0 °C (68.0 °F, teplo)`. Převod ověřen ručně:
  $20 \cdot 9/5 + 32 = 68$.
- **Funguje:** `kelviny` je 293.15 — $20 + 273.15$.
- **Funguje:** zápis přes `t.fahrenheity = 32` přepočte na 0 °C. Setter volá **setter Celsia**,
  takže validace platí i pro zápis ve Fahrenheitech — ověřeno: `-500 °F` → `ValueError`.
- **Klíčové:** validace platí **i v konstruktoru**, protože `self.celsia = celsia`
  v `__init__` projde setterem. To je hlavní důvod, proč se property používá.
- **Funguje:** `kelviny` nemá setter → zápis dá `AttributeError`. Je to záměr, kelviny
  jsou odvozená hodnota.
- **Hraniční:** `-273.15` přesně projde, `-273.16` už ne. Nečíselný vstup → `TypeError`.
- **Omezení:** porovnání `hodnota < ABSOLUTNI_NULA` je na `float`, takže u hodnot
  extrémně blízkých hranici může rozhodovat zaokrouhlení. Prakticky nevadí.

### Na co se doptají

- **Proč getter nesmí vracet `self.celsia`?** Protože `celsia` **je** ta property —
  čtení spustí getter znovu a skončí to `RecursionError`.
- **Proč podtržítko u `_celsia`?** Konvence „interní, nesahej na to". Python to nevynucuje,
  ale odlišuje to skutečný atribut od property.
- **Kdy property a kdy obyčejný atribut?** Property, když potřebuješ **validaci při zápisu**
  nebo **hodnotu počítat za běhu**. Jinak stačí obyčejný atribut — Python nemá důvod
  psát gettery a settery ke všemu jako Java.
- **Co když property nemá setter?** Je **jen ke čtení**, zápis dá `AttributeError`.
  To je přesně případ `kelviny` a `fahrenheity` v původním kódu.
- **Proč se validace píše do setteru a ne do `__init__`?** Aby platila i při **pozdější změně**.
  Kdyby byla jen v konstruktoru, `t.celsia = -500` by prošlo.

---

## Úloha 6 — Kruhový buffer

*Archetyp: iterátor bez StopIteration (katalog #10, #11, #12)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`__iter__` vrací `self.polozky`** (seznam), ne iterátor. Naštěstí to funguje, ale `__next__` se **vůbec nepoužije**. |
| 2 | sémantická | **`__next__` nemá `StopIteration`.** Po posledním prvku vyhodí `IndexError` místo řádného konce. |
| 3 | sémantická | `__iter__` vrací **sdílený stav** — po opravě na `return self` by druhá iterace pokračovala od konce první. |

**Co kód udělá:** vypíše `a b c` — protože `__iter__` vrátí seznam a ten se iteruje sám.
`__next__` je mrtvý kód. Jakmile `__iter__` opravíš na `return self`, projeví se chyba 2
jako `IndexError`.

**Poučení:** to, že kód dává správný výsledek, **neznamená, že je správně**.

### Opravené a rozšířené řešení

In [ ]:
class Buffer:
    """Buffer s omezenou kapacitou; při přetečení vypadne nejstarší položka."""

    def __init__(self, polozky=None, kapacita=5):
        if kapacita < 1:
            raise ValueError(f"kapacita musí být aspoň 1, dostal jsem {kapacita}")
        self.kapacita = kapacita
        self.polozky = list(polozky) if polozky else []    # NIKDY měnitelný default
        if len(self.polozky) > kapacita:
            self.polozky = self.polozky[-kapacita:]        # nechat jen nejnovější

    def pridej(self, polozka):
        """Přidá položku; při přetečení vyhodí nejstarší."""
        self.polozky.append(polozka)
        if len(self.polozky) > self.kapacita:
            self.polozky.pop(0)
        return self

    def __len__(self):
        return len(self.polozky)

    def __contains__(self, x):
        return x in self.polozky

    def __repr__(self):
        return f"Buffer({self.polozky!r}, kapacita={self.kapacita})"

    def __iter__(self):
        """Generátor — každá iterace má VLASTNÍ stav."""
        for polozka in self.polozky:
            yield polozka

In [ ]:
# varianta s explicitním __next__ — kdyby ji zadání chtělo
class BufferIter:
    """Totéž, ale s explicitním __iter__ + __next__."""

    def __init__(self, polozky):
        self.polozky = list(polozky)

    def __iter__(self):
        self._i = 0          # reset při KAŽDÉ nové iteraci
        return self          # jsem sám sobě iterátorem

    def __next__(self):
        if self._i >= len(self.polozky):
            raise StopIteration        # POVINNÉ
        hodnota = self.polozky[self._i]
        self._i += 1
        return hodnota


b = Buffer(["a", "b", "c"], kapacita=3)
print("iterace:", list(b))
print("len:", len(b), "| 'b' in b:", "b" in b, "| 'z' in b:", "z" in b)

b.pridej("d")
print("po přidání 'd' (kapacita 3):", b)     # 'a' vypadlo

print()
print("BufferIter:", list(BufferIter(["x", "y"])))

print()
# rozdíl ve sdílení stavu
bi = BufferIter(["a", "b", "c"])
i1, i2 = iter(bi), iter(bi)
print("s __next__:  ", next(i1), next(i2), " <- sdílejí stav (druhý pokračuje)")

bg = Buffer(["a", "b", "c"])
g1, g2 = iter(bg), iter(bg)
print("s generátorem:", next(g1), next(g2), " <- nezávislé")

print()
print("dvojí průchod týmž bufferem:")
print("  první: ", list(bg))
print("  druhý: ", list(bg), "  <- generátor umí opakovat")

### Výsledky ladění

- **Funguje:** `list(Buffer(["a","b","c"]))` dá `['a', 'b', 'c']` a **řádně skončí**.
  Původní kód s opraveným `__iter__` by spadl na `IndexError`.
- **Funguje:** `len()` a `in` díky `__len__` a `__contains__`.
- **Funguje:** kapacita — po přidání `'d'` do tříprvkového bufferu vypadne `'a'`.
- **Klíčový rozdíl:** generátor umožní **opakovanou iteraci** (`list(bg)` dvakrát dá totéž),
  kdežto verze s `return self` sdílí stav a druhá iterace pokračuje tam, kde první skončila.
  Ověřeno dvěma souběžnými iterátory.
- **Hraniční:** prázdný buffer → `list()` je `[]`, `len` je 0, cyklus neproběhne.
  Kapacita 0 nebo záporná → `ValueError`.
- **Omezení:** `pop(0)` je $O(n)$ — pro velký buffer by se použil `collections.deque`
  s $O(1)$ na obou koncích. Teorie fronty v [SZZTP okruh 1](../../../SZZTP/01-abstraktni-kolekce/).
- **Omezení:** `polozky=None` a `list(...)` uvnitř je nutné — měnitelný default
  by byl sdílený mezi instancemi.

### Na co se doptají

- **Co musí vracet `__iter__`?** **Iterátor**, tedy objekt s `__next__`. Když vrátíš seznam,
  Python si z něj iterátor udělá sám, ale tvoje `__next__` se nikdy nepoužije.
- **Proč je `StopIteration` povinná?** Je to signál „konec", který `for` odchytí.
  Bez ní `for` cyklus nikdy neskončí (nebo spadne na `IndexError`).
- **Jaký je rozdíl mezi iterovatelným a iterátorem?** **Iterovatelné** má `__iter__`
  (seznam, řetězec). **Iterátor** má navíc `__next__` a pamatuje si pozici.
  Seznam je iterovatelný, ale není iterátor — proto ho jde procházet opakovaně.
- **Proč je generátor lepší?** Drží stav sám, každé volání `__iter__` vytvoří **nový** —
  takže jde iterovat opakovaně i souběžně. A nepotřebuje `StopIteration`.
- **Kdy použít `__next__` místo generátoru?** Když to zadání výslovně chce,
  nebo když potřebuješ iterátor s dalšími metodami (reset, peek).
- **Co je `collections.deque`?** Oboustranná fronta s $O(1)$ vkládáním i mazáním na obou koncích.
  Pro kruhový buffer má navíc `maxlen`, který dělá přesně tohle omezení kapacity.

---